# 🏇 KEIBA-AI 金曜予想ノート v6

**実行タイミング**：金曜夜 or 土曜朝（複数回実行OK・都度最新オッズで再予想）

| セル | 内容 |
|---|---|
| セル1 | セットアップ |
| セル2 | 強制アップデート |
| セル3 | 設定・初期化 |
| セル4 | ★ 土曜レース取得・予想・保存 |


In [ ]:
# ── セル1: セットアップ（毎回実行） ──────────────────────────────────
!pip install -q requests beautifulsoup4 lxml pandas

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, requests, pickle, statistics
from datetime import datetime, timezone, timedelta

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
for d in [BASE_DIR, f'{BASE_DIR}/data', f'{BASE_DIR}/app']:
    os.makedirs(d, exist_ok=True)

JST = timezone(timedelta(hours=9))
jst_now = datetime.now(JST)
print(f'✅ セットアップ完了 ({jst_now.strftime("%Y-%m-%d %H:%M:%S")} JST)')


In [ ]:
# ── セル2: 強制アップデート（毎回実行） ─────────────────────────────
import urllib.request

BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
SRC_FILES = [
    'src/__init__.py',
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/bias.py',
    'src/features/__init__.py',
    'src/features/engine.py',
    'src/utils/__init__.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/utils/jst.py',
    'src/scraper/__init__.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/scraper/calendar.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/calibration_xgb.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
]
for rel in SRC_FILES:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    print(f'✅ {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
sys.path.insert(0, BASE_DIR)
print('🔄 アップデート完了')

In [ ]:
# ── セル3: 設定・初期化（毎回実行） ─────────────────────────────────
# ▼▼ 賭け金はここで変更 ▼▼
BANKROLL    = 100_000
TOP_N_RACES = 6
FUKU_AMT = 500   # 複勝
TAN_AMT  = 300   # 単勝
WIDE_AMT = 300   # ワイド
REN_AMT  = 300   # 馬連
TAN2_AMT = 300   # 馬単
SAN_AMT  = 300   # 三連複

# モジュール読み込み
from src.features.engine import init_engine
from src.utils.db import (init_db, save_race_db, save_bets_db,
                           save_history_db, save_results_db, check_and_update_bets)
from src.betting.make_bets import init_betting, make_bets, log_bet_simulation
from src.betting.ev_filter import ability_first_loose
from src.betting.app_json import to_app_json
from src.scraper.jra_scraper import fetch_races_on_date, fetch_results

HEADERS  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE = 'https://www.jra.go.jp'
HIST_PATH = f'{BASE_DIR}/data/history.db'
BIAS_PATH = f'{BASE_DIR}/data/track_bias_latest.json'

# DB初期化・エンジン初期化・賭け金初期化
init_db(BASE_DIR)
init_engine(BASE_DIR)
init_betting(BASE_DIR, bankroll=BANKROLL,
             fuku_amt=FUKU_AMT, tan_amt=TAN_AMT, wide_amt=WIDE_AMT,
             ren_amt=REN_AMT, tan2_amt=TAN2_AMT, san_amt=SAN_AMT)

# バイアス読み込み（前週分・なければ None）
avg_bias = None
if os.path.exists(BIAS_PATH):
    with open(BIAS_PATH) as f:
        avg_bias = json.load(f)
    print(f'  📊 バイアス({avg_bias.get("date","")}): {avg_bias.get("summary","フラット")}')
else:
    print('  📊 バイアス: なし（フラット想定）')

print(f'✅ 初期化完了  複勝¥{FUKU_AMT} 単勝¥{TAN_AMT} ワイド¥{WIDE_AMT}')


In [ ]:
# ── セル4: ★ 土曜レース取得・予想・保存 ──────────────────────────
# 金曜夜 → 土曜の開催日を自動計算
# 土曜朝 → 当日として取得（複数回実行で最新オッズに追従）

jst_now = datetime.now(JST)
weekday = jst_now.weekday()  # 4=金 5=土 6=日

from datetime import timedelta as _td
if weekday == 4:    # 金曜実行 → 翌日（土曜）
    target_date = (jst_now + _td(days=1)).strftime('%Y%m%d')
elif weekday == 5:  # 土曜実行 → 当日
    target_date = jst_now.strftime('%Y%m%d')
else:               # その他（手動上書き）
    target_date = jst_now.strftime('%Y%m%d')

print(f'📅 取得日: {target_date}  ({["月","火","水","木","金","土","日"][weekday]}曜)')

# セッション作成（JRAログイン不要・パブリックページ）
sess = requests.Session()
sess.headers.update(HEADERS)
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

# レース取得
races = fetch_races_on_date(sess, target_date, HIST_PATH)
print(f'📋 取得レース: {len(races)}R')

# 予想生成
selected = ability_first_loose(races, avg_bias, top_n=TOP_N_RACES)
print(f'⭐ 厳選: {len(selected)}レース\n')

if not selected:
    print('⚠ 厳選レースなし（EV基準を満たすレースがありません）')
    print('  → EV_THRESHOLD / WIN_PROB_MIN を確認してください')
    print('  → 出馬表が未公開の場合はオッズなしのため予想不可')

# 予想表示・DB保存
total_inv = 0
for i, c in enumerate(selected, 1):
    race = c['race']; top1 = c['top1']; scored = c['scored']
    bets = make_bets(c)
    invest = sum(b['amount'] for b in bets)
    total_inv += invest

    print(f'【{i}】{race["racecourse"]} R{race["race_num"]:02d} {race.get("race_name","")}'
          f'  {race["distance"]}m{race["surface"]}  {race.get("num_horses",0)}頭')
    print(f'  ◎ #{top1["num"]} {top1["name"]}  {top1.get("win_odds",0):.1f}倍  スコア:{top1["total"]:.2f}')
    for b in bets:
        print(f'  {b["type"]} #{b["nums"][0]} ¥{b["amount"]:,}  EV:{b["ev"]:.2f}')
    print(f'  投資: ¥{invest:,}\n')

    save_race_db(race, BASE_DIR)
    save_bets_db(race['date'], race['id'], bets, BASE_DIR)
    log_bet_simulation(race['date'], c, BASE_DIR)

print(f'💰 投資合計: ¥{total_inv:,}')

# アプリ用JSON保存
jst_now = datetime.now(JST)
app_data = to_app_json(selected, races, avg_bias, jst_now, day_type='saturday')
app_path = f'{BASE_DIR}/app/latest.json'
with open(app_path, 'w', encoding='utf-8') as f:
    json.dump(app_data, f, ensure_ascii=False, indent=2)
print(f'✅ アプリJSON保存: {app_path}')
